# DRGS: Depth-Regularized 3D Gaussian Splatting (CVPRW 2024)

**Paper**: [Depth-Regularized Optimization for 3D Gaussian Splatting in Few-Shot Images](https://arxiv.org/abs/2311.13398)

저자의 원래 파이프라인을 그대로 사용합니다:
1. **COLMAP sparse points** → 각 카메라에 project → sparse depth map
2. **ZoeDepth** → monocular depth 추정 (dense)
3. **optimize_depth()** → ZoeDepth를 COLMAP에 맞춰 scale/shift 정합
4. 정합된 depth로 **depth supervision + depth regularization** 학습

---
## Section 0: Configuration

In [ ]:
import os

PROJECT_ROOT = "/content/DepthRegularizedGS"
REPO_URL = "https://github.com/BAEJUNHAK/DepthRegularizedGS.git"

DRIVE_MOUNT = "/content/drive"
DRIVE_SAVE_DIR = os.path.join(DRIVE_MOUNT, "MyDrive", "DRGS_results")

# 데이터 Google Drive file IDs
RGB_GDRIVE_ID = "1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB"
DEPTH_GDRIVE_ID = "1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ"

# raw 데이터 (calib INI 형식) → COLMAP 변환 후 저장 경로
SCENE_NAME = "mitsubishi"
RAW_DIR = os.path.join(PROJECT_ROOT, "data", f"{SCENE_NAME}_raw")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", SCENE_NAME)  # COLMAP 포맷

# 변환 파라미터
TRAIN_RATIO = 0.8
SPLIT_SEED = 42
DEPTH_SUBSAMPLE = 50  # sparse point 생성 시 픽셀 간격

# 학습 하이퍼파라미터
KSHOT = 30
SEED = 3
RESOLUTION = 1
ITERATIONS = 30000

TEST_ITERS = "7000 15000 30000"
SAVE_ITERS = "7000 15000 30000"

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
BASELINE_DIR = os.path.join(OUTPUT_DIR, f"{SCENE_NAME}_baseline_k{KSHOT}_s{SEED}")
DRGS_DIR = os.path.join(OUTPUT_DIR, f"{SCENE_NAME}_drgs_k{KSHOT}_s{SEED}")

print(f"Scene: {SCENE_NAME}")
print(f"K-shot: {KSHOT}, Seed: {SEED}")

---
## Section 1: Environment Setup

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU 없음")
print(f"GPU: {torch.cuda.get_device_name(0)}, CUDA: {torch.version.cuda}, PyTorch: {torch.__version__}")

In [ ]:
import os, subprocess

if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} --recursive {PROJECT_ROOT}
else:
    print(f"이미 존재: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
!git submodule update --init --recursive

# ── CUDA 빌드 환경 설정 ──
!pip install -q ninja

cuda_path = "/usr/local/cuda"
if os.path.exists(cuda_path):
    os.environ["CUDA_HOME"] = cuda_path

# GPU 아키텍처 자동 감지 → TORCH_CUDA_ARCH_LIST 설정
import torch
capability = torch.cuda.get_device_capability()
arch = f"{capability[0]}.{capability[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = arch
print(f"CUDA_HOME={cuda_path}, TORCH_CUDA_ARCH_LIST={arch}, GPU={torch.cuda.get_device_name()}")

# ── diff-gaussian-rasterization ──
!pip install --no-build-isolation submodules/diff-gaussian-rasterization-depth-acc

# ── simple-knn (pip 실패 시 setup.py install로 fallback) ──
ret = os.system("pip install --no-build-isolation submodules/simple-knn")
if ret != 0:
    print("\n[WARNING] pip install 실패 — setup.py install로 재시도...")
    os.chdir(os.path.join(PROJECT_ROOT, "submodules", "simple-knn"))
    !python setup.py install 2>&1 | tail -30
    os.chdir(PROJECT_ROOT)

# ── 추가 의존성 ──
!pip install -q plyfile==0.8.1 imageio lpips opencv-python-headless

In [ ]:
# 설치 검증
import torch
from diff_gaussian_rasterization_depth_acc import GaussianRasterizer
from simple_knn._C import distCUDA2
import plyfile, cv2
print("모든 의존성 OK")

---
## Section 2: Data Preparation

1. raw 데이터 다운로드 (calib INI + RGB + depth PNG)
2. **COLMAP 바이너리 포맷으로 변환** (`convert_custom_to_colmap.py`)
   - GT depth에서 sparse 3D points 생성 (COLMAP이 만든 것처럼)
   - `cameras.bin`, `images.bin`, `points3D.bin` 생성
3. 이후 저자의 원래 코드가 그대로 동작:
   - `refineColmapWithIndex` → sparse points 필터링
   - `readColmapCameras` → ZoeDepth + `optimize_depth()` → refined depth

### 2.1 Google Drive 마운트 & 데이터 다운로드

In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

In [ ]:
os.chdir(PROJECT_ROOT)
os.makedirs(RAW_DIR, exist_ok=True)

!pip install -q gdown

!gdown "{RGB_GDRIVE_ID}" -O /tmp/dataset.zip
!unzip -q -o /tmp/dataset.zip -d /tmp/rgb_extract

!gdown "{DEPTH_GDRIVE_ID}" -O /tmp/depth_data.zip
!unzip -q -o /tmp/depth_data.zip -d /tmp/depth_extract

!cp /tmp/rgb_extract/dataset/rgb_*.png {RAW_DIR}/
!cp /tmp/rgb_extract/dataset/calib_*.ini {RAW_DIR}/
!cp /tmp/depth_extract/dataset_depth/depth_raw_*.png {RAW_DIR}/

!rm -rf /tmp/dataset.zip /tmp/depth_data.zip /tmp/rgb_extract /tmp/depth_extract

import glob
print(f"RGB: {len(glob.glob(os.path.join(RAW_DIR, 'rgb_*.png')))}")
print(f"Depth: {len(glob.glob(os.path.join(RAW_DIR, 'depth_raw_*.png')))}")
print(f"Calib: {len(glob.glob(os.path.join(RAW_DIR, 'calib_*.ini')))}")

### 2.2 COLMAP 포맷으로 변환

GT depth에서 sparse 3D points를 추출하여 COLMAP이 만든 것처럼 `cameras.bin`, `images.bin`, `points3D.bin`을 생성합니다.

In [ ]:
os.chdir(PROJECT_ROOT)

!python scripts/convert_custom_to_colmap.py \
    --input_dir {RAW_DIR} \
    --output_dir {DATA_DIR} \
    --train_ratio {TRAIN_RATIO} \
    --split_seed {SPLIT_SEED} \
    --depth_subsample {DEPTH_SUBSAMPLE}

### 2.3 변환 결과 검증

In [ ]:
import json, glob
from PIL import Image
import matplotlib.pyplot as plt

# COLMAP 구조 확인
checks = {
    "images/":       os.path.isdir(os.path.join(DATA_DIR, "images")),
    "sparse/0/":     os.path.isdir(os.path.join(DATA_DIR, "sparse", "0")),
    "cameras.bin":   os.path.isfile(os.path.join(DATA_DIR, "sparse", "0", "cameras.bin")),
    "images.bin":    os.path.isfile(os.path.join(DATA_DIR, "sparse", "0", "images.bin")),
    "points3D.bin":  os.path.isfile(os.path.join(DATA_DIR, "sparse", "0", "points3D.bin")),
    "split_index.json": os.path.isfile(os.path.join(DATA_DIR, "split_index.json")),
}
for k, v in checks.items():
    status = "OK" if v else "MISSING"
    print(f"  [{status}] {k}")

with open(os.path.join(DATA_DIR, "split_index.json")) as f:
    split = json.load(f)
print(f"\nTrain: {len(split['train'])}, Test: {len(split['test'])}")
print(f"Images: {len(glob.glob(os.path.join(DATA_DIR, 'images', '*.png')))}")

# 샘플 이미지
imgs = sorted(glob.glob(os.path.join(DATA_DIR, "images", "*.png")))[:5]
fig, axes = plt.subplots(1, len(imgs), figsize=(3*len(imgs), 3))
for ax, p in zip(axes, imgs):
    ax.imshow(Image.open(p))
    ax.set_title(os.path.basename(p), fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## Section 3: Training

**저자의 원래 파이프라인 그대로 실행합니다:**
- COLMAP sparse points → 각 카메라에 project
- ZoeDepth monocular depth 추정 (첫 실행 시 가중치 자동 다운로드 ~400MB)
- `optimize_depth()`: ZoeDepth를 COLMAP points에 scale/shift 정합
- 정합된 depth로 supervision + regularization

In [ ]:
os.chdir(PROJECT_ROOT)
os.makedirs("debug", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

### 3.1 Baseline 3DGS (depth 없이)

In [ ]:
os.chdir(PROJECT_ROOT)

!python train.py \
    -s {DATA_DIR} \
    --eval \
    --port 6009 \
    --model_path {BASELINE_DIR} \
    --resolution {RESOLUTION} \
    --kshot {KSHOT} \
    --seed {SEED} \
    --iterations {ITERATIONS} \
    --test_iterations {TEST_ITERS} \
    --save_iterations {SAVE_ITERS}

### 3.2 DRGS (--depth --usedepthReg)

저자의 방법: COLMAP sparse depth → ZoeDepth 정합 → depth supervision + Canny regularization

In [ ]:
os.chdir(PROJECT_ROOT)

!python train.py \
    -s {DATA_DIR} \
    --eval \
    --port 6010 \
    --model_path {DRGS_DIR} \
    --resolution {RESOLUTION} \
    --kshot {KSHOT} \
    --seed {SEED} \
    --iterations {ITERATIONS} \
    --test_iterations {TEST_ITERS} \
    --save_iterations {SAVE_ITERS} \
    --depth \
    --usedepthReg

### 3.3 TensorBoard (선택)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

---
## Section 4: Rendering & Evaluation

In [ ]:
os.chdir(PROJECT_ROOT)

!python render.py -s {DATA_DIR} -m {BASELINE_DIR}
!python render.py -s {DATA_DIR} -m {DRGS_DIR}

In [ ]:
os.chdir(PROJECT_ROOT)

!python metrics.py -m {BASELINE_DIR} {DRGS_DIR}

### 결과 비교

In [ ]:
import json, pandas as pd

results = {}
for label, d in [("Baseline 3DGS", BASELINE_DIR), ("DRGS", DRGS_DIR)]:
    rp = os.path.join(d, "results.json")
    if os.path.exists(rp):
        with open(rp) as f:
            for _, m in json.load(f).items():
                results[label] = m

if results:
    df = pd.DataFrame(results).T[["PSNR","SSIM","LPIPS"]]
    df["PSNR"] = df["PSNR"].map("{:.2f}".format)
    df["SSIM"] = df["SSIM"].map("{:.4f}".format)
    df["LPIPS"] = df["LPIPS"].map("{:.4f}".format)
    print(f"\n=== {SCENE_NAME} | K={KSHOT} | Seed={SEED} ===\n")
    display(df)

---
## Section 5: Visualization

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob, os

def find_render_dir(model_dir):
    test_dir = os.path.join(model_dir, "test")
    if not os.path.isdir(test_dir): return None
    ours = sorted(glob.glob(os.path.join(test_dir, "ours_*")))
    return ours[-1] if ours else None

brd = find_render_dir(BASELINE_DIR)
drd = find_render_dir(DRGS_DIR)

if brd and drd:
    gts = sorted(glob.glob(os.path.join(brd, "gt", "*.png")))
    bfs = sorted(glob.glob(os.path.join(brd, "renders", "*.png")))
    dfs = sorted(glob.glob(os.path.join(drd, "renders", "*.png")))

    n = min(5, len(gts))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    if n == 1: axes = axes.reshape(3, 1)
    for i in range(n):
        axes[0,i].imshow(Image.open(gts[i])); axes[0,i].set_title(f"GT #{i}"); axes[0,i].axis("off")
        axes[1,i].imshow(Image.open(bfs[i])); axes[1,i].set_title(f"Baseline #{i}"); axes[1,i].axis("off")
        axes[2,i].imshow(Image.open(dfs[i])); axes[2,i].set_title(f"DRGS #{i}"); axes[2,i].axis("off")
    for r, lbl in enumerate(["GT","Baseline","DRGS"]):
        axes[r,0].set_ylabel(lbl, fontsize=12, rotation=0, labelpad=60, va="center")
    plt.suptitle(f"Test Views — {SCENE_NAME} K={KSHOT}", fontsize=14)
    plt.tight_layout(); plt.show()

In [ ]:
# ZoeDepth debug 이미지 확인 (저자 파이프라인의 depth 정합 결과)
import glob
from PIL import Image
import matplotlib.pyplot as plt

debug_dir = os.path.join(PROJECT_ROOT, "debug")
source_files = sorted(glob.glob(os.path.join(debug_dir, "*_source.png")))
refined_files = sorted(glob.glob(os.path.join(debug_dir, "*_refined.png")))
target_files = sorted(glob.glob(os.path.join(debug_dir, "*_target.png")))

if source_files:
    n = min(5, len(source_files))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 10))
    if n == 1: axes = axes.reshape(3, 1)
    for i in range(n):
        axes[0,i].imshow(Image.open(source_files[i])); axes[0,i].set_title(f"ZoeDepth #{i}"); axes[0,i].axis("off")
        if i < len(refined_files): axes[1,i].imshow(Image.open(refined_files[i]))
        axes[1,i].set_title(f"Refined #{i}"); axes[1,i].axis("off")
        if i < len(target_files): axes[2,i].imshow(Image.open(target_files[i]), cmap="gray")
        axes[2,i].set_title(f"COLMAP target #{i}"); axes[2,i].axis("off")
    axes[0,0].set_ylabel("ZoeDepth\n(source)", fontsize=10, rotation=0, labelpad=70, va="center")
    axes[1,0].set_ylabel("Optimized\n(refined)", fontsize=10, rotation=0, labelpad=70, va="center")
    axes[2,0].set_ylabel("COLMAP\n(target)", fontsize=10, rotation=0, labelpad=70, va="center")
    plt.suptitle("ZoeDepth → COLMAP Depth Optimization (저자 파이프라인)", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print("debug/ 이미지 없음. DRGS 학습(--depth) 실행 후 생성됩니다.")

---
## Section 6: 결과 저장

In [ ]:
import shutil

exp_name = f"{SCENE_NAME}_k{KSHOT}_s{SEED}"
save_dir = os.path.join(DRIVE_SAVE_DIR, exp_name)
os.makedirs(save_dir, exist_ok=True)

for label, src in [("baseline", BASELINE_DIR), ("drgs", DRGS_DIR)]:
    dst = os.path.join(save_dir, label)
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"[OK] {label} → {dst}")

# debug 이미지도 복사
debug_src = os.path.join(PROJECT_ROOT, "debug")
if os.path.exists(debug_src) and os.listdir(debug_src):
    debug_dst = os.path.join(save_dir, "debug")
    if os.path.exists(debug_dst): shutil.rmtree(debug_dst)
    shutil.copytree(debug_src, debug_dst)
    print(f"[OK] debug → {debug_dst}")

print(f"\n저장 완료: {save_dir}")